In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import db_dtypes

In [2]:
client = bigquery.Client(project='first-project-321219')

query = """
SELECT 
    u.id AS user_id,
    u.age,
    u.gender,
    u.country,
    -- FEATURE ENGINEERING: Past Behavior
    COUNT(DISTINCT e.id) AS total_web_sessions,
    COUNT(DISTINCT o.order_id) AS total_orders,
   -- SUM(oi.sale_price) AS total_historic_spend,
    AVG(oi.sale_price) AS avg_order_value,
   
    
    -- THE TARGET: Total Spend (What we want to predict)
    SUM(oi.sale_price) AS label_total_spend 

FROM `bigquery-public-data.thelook_ecommerce.users` u
LEFT JOIN `bigquery-public-data.thelook_ecommerce.orders` o 
    ON u.id = o.user_id AND o.status NOT IN ('Cancelled', 'Returned')
LEFT JOIN `bigquery-public-data.thelook_ecommerce.order_items` oi 
    ON o.order_id = oi.order_id
LEFT JOIN `bigquery-public-data.thelook_ecommerce.events` e 
    ON u.id = e.user_id

WHERE u.created_at >= '2022-01-01' -- Filter for a specific cohort
GROUP BY 1, 2, 3, 4
"""

df = client.query(query).to_dataframe()
print(df.head())

   user_id  age gender country  total_web_sessions  total_orders  \
0     7443   20      F  Brasil                  28             0   
1    48692   69      F   China                  35             0   
2    31218   15      M   China                  44             0   
3    55790   23      F   China                  66             0   
4    27218   36      M   China                  30             0   

   avg_order_value  label_total_spend  
0              NaN                NaN  
1              NaN                NaN  
2              NaN                NaN  
3              NaN                NaN  
4              NaN                NaN  


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58857 entries, 0 to 58856
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             58857 non-null  Int64  
 1   age                 58857 non-null  Int64  
 2   gender              58857 non-null  object 
 3   country             58857 non-null  object 
 4   total_web_sessions  58857 non-null  Int64  
 5   total_orders        58857 non-null  Int64  
 6   avg_order_value     38900 non-null  float64
 7   label_total_spend   38900 non-null  float64
dtypes: Int64(4), float64(2), object(2)
memory usage: 3.8+ MB


In [4]:
def load_data(dataframe, datefrom = None, dateto = None):
    
    if not isinstance(dataframe, pd.DataFrame):   #isinstance(): check specified object is in specificed type.
        print('Error: Input is not a DataFrame')
        return None
  

    if 'label_total_spend' not in dataframe.columns: 
        print(f'Column Error: label_total_spend is not in DataFrame')
        return dataframe
        
    # try:
    #     dataframe['cart_created_date'] = pd.to_datetime(dataframe['cart_created_date'])
    # except Exception:
    #     print(f'Date Conversion Error: {Exception}')

    return dataframe

    

In [6]:
import os
os.getcwd()

'C:\\Users\\ipnga'